In [31]:
# ============================================================
# Install dependencies
# ============================================================
!pip install -q chromadb rank_bm25 sentence-transformers openai


In [32]:
# ============================================================
# Mount Google Drive (to persist ChromaDB index)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/dup_detect/data/chroma_db', exist_ok=True)
os.makedirs('/content/drive/MyDrive/dup_detect/results', exist_ok=True)

DRIVE_DIR   = '/content/drive/MyDrive/dup_detect'
CHROMA_DIR  = f'{DRIVE_DIR}/data/chroma_db'
RESULTS_DIR = f'{DRIVE_DIR}/results'

print("Drive mounted. Folders ready:")
print(f"  {DRIVE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Folders ready:
  /content/drive/MyDrive/dup_detect


In [33]:
# ============================================================
# Upload  files
# ============================================================
# Run this cell to upload the 5 XML files + CSV
# You only need to do this once — they stay in Drive

from google.colab import files

print("Upload your 5 XML files + full-edkII-dd.csv")
uploaded = files.upload()

# Move uploaded files to Drive
import shutil
for fname in uploaded:
    src = f'/content/{fname}'
    dst = f'{DRIVE_DIR}/data/{fname}'
    shutil.move(src, dst)
    print(f"  Saved: {dst}")


Upload your 5 XML files + full-edkII-dd.csv


Saving full-edkII-dd.csv to full-edkII-dd.csv
Saving tianocore_closed_0_1000.xml to tianocore_closed_0_1000.xml
Saving tianocore_closed_1000_2000.xml to tianocore_closed_1000_2000.xml
Saving tianocore_closed_2000_3000.xml to tianocore_closed_2000_3000.xml
Saving tianocore_closed_3000_4000.xml to tianocore_closed_3000_4000.xml
Saving tianocore_closed_4000_5000.xml to tianocore_closed_4000_5000.xml
  Saved: /content/drive/MyDrive/dup_detect/data/full-edkII-dd.csv
  Saved: /content/drive/MyDrive/dup_detect/data/tianocore_closed_0_1000.xml
  Saved: /content/drive/MyDrive/dup_detect/data/tianocore_closed_1000_2000.xml
  Saved: /content/drive/MyDrive/dup_detect/data/tianocore_closed_2000_3000.xml
  Saved: /content/drive/MyDrive/dup_detect/data/tianocore_closed_3000_4000.xml
  Saved: /content/drive/MyDrive/dup_detect/data/tianocore_closed_4000_5000.xml


In [34]:
# ============================================================
# Config & API keys
# ============================================================

import os
from google.colab import userdata

# ── OpenAI key ────────────────────────────────────────────────
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("OpenAI API key loaded")
except:
    OPENAI_API_KEY = input("Paste your OpenAI API key: ").strip()

# ── Anthropic key (for Claude models) ────────────────────────
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("Anthropic API key loaded")
except:
    ANTHROPIC_API_KEY = ""
    print("No Anthropic API key — Claude models will be skipped")

# ── Org ID for OpenAI opt-out header ─────────────────────────
try:
    OPENAI_ORG_ID = userdata.get('OPENAI_ORG_ID')
    print(f"OpenAI org opt-out header: set")
except:
    OPENAI_ORG_ID = ""
    print("OpenAI org opt-out header: not set")

# ── All models to run ─────────────────────────────────────────
# Each model will run System A and System B independently
# Results saved as separate CSVs per model in RESULTS_DIR
OPENAI_MODELS = [
    "gpt-4o-mini",    # Chat Completions
    "gpt-4o",         # Chat Completions
    "gpt-4.1-mini",   # Chat Completions
    "gpt-4.1",        # Chat Completions
    "gpt-5.4-mini",   # Responses API
    "gpt-5.4",        # Responses API
    "gpt-5.5",        # Responses API
]

CLAUDE_MODELS = [
    "claude-haiku-4-5",       # fastest, cheapest
    "claude-sonnet-4-6",      # balanced best price/performance
    "claude-opus-4-7",        # flagship most capable (Apr 2026)
] if ANTHROPIC_API_KEY else []

ALL_MODELS = OPENAI_MODELS + CLAUDE_MODELS

print(f"\nModels to run ({len(ALL_MODELS)} total):")
for m in ALL_MODELS:
    provider = "Anthropic" if m.startswith("claude") else "OpenAI"
    print(f"  [{provider}] {m}")

XML_FILES = [
    f'{DRIVE_DIR}/data/tianocore_closed_0_1000.xml',
    f'{DRIVE_DIR}/data/tianocore_closed_1000_2000.xml',
    f'{DRIVE_DIR}/data/tianocore_closed_2000_3000.xml',
    f'{DRIVE_DIR}/data/tianocore_closed_3000_4000.xml',
    f'{DRIVE_DIR}/data/tianocore_closed_4000_5000.xml',
]
GITHUB_CSV      = f'{DRIVE_DIR}/data/full-edkII-dd.csv'
XML_CONTEXT_K   = 3
DUP_CANDIDATE_K = 5

print("\nConfig ready.")


OpenAI API key loaded
Anthropic API key loaded
OpenAI org opt-out header: set

Models to run (10 total):
  [OpenAI] gpt-4o-mini
  [OpenAI] gpt-4o
  [OpenAI] gpt-4.1-mini
  [OpenAI] gpt-4.1
  [OpenAI] gpt-5.4-mini
  [OpenAI] gpt-5.4
  [OpenAI] gpt-5.5
  [Anthropic] claude-haiku-4-5
  [Anthropic] claude-sonnet-4-6
  [Anthropic] claude-opus-4-7

Config ready.


In [35]:
# ============================================================
# CELL 5 — Ingest: parse XML corpus + GitHub issues
# ============================================================

import re
import xml.etree.ElementTree as ET
import pandas as pd


def _clean(text):
    text = re.sub(r"https?://\S+", "<URL>", text or "")
    return re.sub(r"\s+", " ", text).strip()


def _xml_body(bug_el):
    bodies = []
    for ld in bug_el.findall("long_desc"):
        t = ld.find("thetext")
        if t is not None and t.text:
            bodies.append(t.text.strip())
        if len(bodies) >= 3:
            break
    return "\n\n".join(bodies)


def load_xml_corpus(xml_paths):
    bugs = []
    for path in xml_paths:
        tree = ET.parse(path)
        root = tree.getroot()
        for bug_el in root.findall("bug"):
            def t(tag):
                n = bug_el.find(tag)
                return (n.text or "").strip() if n is not None else ""
            title      = t("short_desc")
            body       = _xml_body(bug_el)
            resolution = t("resolution")
            dup_node   = bug_el.find("dup_id")
            dup_id     = dup_node.text.strip() if dup_node is not None and dup_node.text else None
            bugs.append(dict(
                id         = t("bug_id"),
                title      = title,
                body       = body,
                package    = t("cf_package"),
                component  = t("component"),
                priority   = t("priority"),
                resolution = resolution,
                dup_id     = dup_id,
                text       = _clean(f"Title: {title}\n\nDescription: {body}"),
            ))
    print(f"[ingest] XML corpus: {len(bugs):,} bugs from {len(xml_paths)} files")
    return bugs


def load_github_issues(csv_path):
    df = pd.read_csv(csv_path)
    issues = []
    for _, row in df.iterrows():
        title    = str(row.get("Title", "") or "")
        package  = str(row.get("package", "") or "")
        itype    = str(row.get("type", "") or "")
        priority = str(row.get("priority", "") or "")
        iid      = str(int(row["Issue Number"]))
        issues.append(dict(
            id       = iid,
            title    = title,
            package  = package,
            type     = itype,
            priority = priority,
            url      = str(row.get("URL", "") or ""),
            text     = _clean(f"Title: {title}\nPackage: {package}\nType: {itype}\nPriority: {priority}"),
        ))
    print(f"[ingest] GitHub issues: {len(issues):,} issues loaded")
    return issues


xml_bugs  = load_xml_corpus(XML_FILES)
gh_issues = load_github_issues(GITHUB_CSV)

from collections import Counter
print(f"\nTop packages (GitHub issues):")
for pkg, n in Counter(i["package"] for i in gh_issues).most_common(6):
    print(f"  {pkg or 'none':35s} {n}")

[ingest] XML corpus: 3,529 bugs from 5 files
[ingest] GitHub issues: 2,610 issues loaded

Top packages (GitHub issues):
  mdemodulepkg                        486
  basetools                           486
  mdepkg                              199
  ueficpupkg                          194
  ovmfpkg                             155
  shellpkg                            131


In [36]:
# ============================================================
# CELL 6 — Build retriever (ChromaDB + BM25)
# Run once — index persists to Drive
# ============================================================

import torch
import chromadb
from chromadb import EmbeddingFunction, Documents, Embeddings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm.notebook import tqdm

EMBED_MODEL    = "BAAI/bge-base-en-v1.5"
EMBED_BATCH    = 64
RRF_K          = 60
XML_COLLECTION = "xml_corpus"
GH_COLLECTION  = "github_issues"


def get_device():
    if torch.cuda.is_available():
        print("[retriever] Using CUDA GPU")
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("[retriever] Using Apple MPS")
        return "mps"
    print("[retriever] Using CPU")
    return "cpu"


def tokenise(text):
    text = re.sub(r"[^a-zA-Z0-9_]", " ", (text or "").lower())
    return [t for t in text.split() if len(t) > 1]


def rrf(ranked_a, ranked_b, k=RRF_K):
    scores = {}
    for rank, doc_id in enumerate(ranked_a, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    for rank, doc_id in enumerate(ranked_b, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.__getitem__, reverse=True)


class BGEEmbeddingFunction(EmbeddingFunction):
    def __init__(self):
        device = get_device()
        print(f"[retriever] Loading {EMBED_MODEL} ...")
        self._model = SentenceTransformer(EMBED_MODEL, device=device)
        print("[retriever] Model loaded")

    def __call__(self, input: Documents) -> Embeddings:
        embeddings = self._model.encode(
            list(input),
            batch_size           = EMBED_BATCH,
            normalize_embeddings = True,
            show_progress_bar    = False,
        )
        return embeddings.tolist()


class Retriever:
    def __init__(self, xml_bugs, github_issues, persist_dir=CHROMA_DIR):
        self.xml_bugs      = xml_bugs
        self.github_issues = github_issues
        self.gh_by_id      = {i["id"]: i for i in github_issues}
        self.xml_by_id     = {b["id"]: b for b in xml_bugs}

        embed_fn = BGEEmbeddingFunction()
        client   = chromadb.PersistentClient(path=persist_dir)

        self._xml_col = client.get_or_create_collection(
            XML_COLLECTION, embedding_function=embed_fn,
            metadata={"hnsw:space": "cosine"},
        )
        self._gh_col = client.get_or_create_collection(
            GH_COLLECTION, embedding_function=embed_fn,
            metadata={"hnsw:space": "cosine"},
        )
        self._xml_bm25     = None
        self._xml_bm25_ids = []
        self._gh_bm25      = None
        self._gh_bm25_ids  = []

    def build(self, force_reindex=False):
        self._index(self._xml_col, self.xml_bugs,      "XML corpus",    force_reindex)
        self._index(self._gh_col,  self.github_issues, "GitHub issues", force_reindex)

        self._xml_bm25_ids = [b["id"] for b in self.xml_bugs]
        self._xml_bm25     = BM25Okapi([tokenise(b["text"]) for b in self.xml_bugs])
        self._gh_bm25_ids  = [i["id"] for i in self.github_issues]
        self._gh_bm25      = BM25Okapi([tokenise(i["text"]) for i in self.github_issues])
        print("[retriever] Both indexes ready")

    def _index(self, col, docs, label, force):
        if col.count() > 0 and not force:
            print(f"[retriever] {label}: {col.count():,} docs already indexed — skipping")
            return
        print(f"[retriever] Embedding {len(docs):,} {label} docs ...")
        for i in tqdm(range(0, len(docs), EMBED_BATCH), desc=label):
            batch = docs[i : i + EMBED_BATCH]
            col.add(
                ids       = [d["id"]   for d in batch],
                documents = [d["text"] for d in batch],
                metadatas = [{"title": d["title"][:500],
                              "package": d.get("package", "")} for d in batch],
            )
        print(f"[retriever] {label}: {col.count():,} docs indexed")

    def get_xml_context(self, issue, k=3):
        return self._hybrid(issue["text"], self._xml_col,
                            self._xml_bm25, self._xml_bm25_ids,
                            self.xml_by_id, None, k)

    def get_xml_dup_pairs(self, issue, k=3):
        """
        Retrieve top-k similar XML bugs and find their duplicate pairs.
        Returns list of (bug, dup_bug) tuples.
        dup_bug is None if no pair found in corpus.
        """
        similar_bugs = self.get_xml_context(issue, k=k)
        pairs = []
        for bug in similar_bugs:
            dup_id = bug.get("dup_id")
            if dup_id and dup_id in self.xml_by_id:
                pairs.append((bug, self.xml_by_id[dup_id]))
            else:
                pairs.append((bug, None))
        return pairs

    def get_dup_candidates(self, issue, k=5):
        return self._hybrid(issue["text"], self._gh_col,
                            self._gh_bm25, self._gh_bm25_ids,
                            self.gh_by_id, issue["id"], k)

    def _hybrid(self, query_text, col, bm25, bm25_ids, index, exclude_id, k):
        fetch     = min(k * 4, col.count())
        dense_res = col.query(query_texts=[query_text], n_results=fetch,
                              include=["distances"])
        dense_ids = [i for i in dense_res["ids"][0] if i != exclude_id]

        scores     = bm25.get_scores(tokenise(query_text))
        sparse_ids = [bm25_ids[i]
                      for i in sorted(range(len(scores)), key=lambda x: -scores[x])
                      if bm25_ids[i] != exclude_id][:fetch]

        merged = rrf(dense_ids, sparse_ids)[:k]
        return [index[i] for i in merged if i in index]


retriever = Retriever(xml_bugs, gh_issues)
retriever.build(force_reindex=False)


[retriever] Using CPU
[retriever] Loading BAAI/bge-base-en-v1.5 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[retriever] Model loaded
[retriever] XML corpus: 3,529 docs already indexed — skipping
[retriever] GitHub issues: 2,610 docs already indexed — skipping
[retriever] Both indexes ready


In [37]:
def predict_A(issue, dup_candidates, model):
    """System A — no XML context"""
    user  = "QUERY ISSUE:\n" + fmt_github(issue)
    user += "\n\nCANDIDATE ISSUES:\n"
    for i, c in enumerate(dup_candidates, 1):
        user += f"\n[{i}]\n{fmt_github(c)}\n"
    return call_llm(_PROMPT_A, user, model)


def predict_B(issue, dup_candidates, xml_pairs, model):
    """System B — with XML duplicate pairs as RAG context"""
    user = ""
    pairs_with_both = [(a, b) for a, b in xml_pairs if b is not None]
    if pairs_with_both:
        user += "DUPLICATE EXAMPLES FROM THIS CODEBASE:\n"
        for bug, dup_bug in pairs_with_both:
            user += f"\n{fmt_xml_pair(bug, dup_bug)}\n"
        user += "\n---\n\n"
    user += "QUERY ISSUE:\n" + fmt_github(issue)
    user += "\n\nCANDIDATE ISSUES (only these can be duplicates of the query):\n"
    for i, c in enumerate(dup_candidates, 1):
        user += f"\n[{i}]\n{fmt_github(c)}\n"
    return call_llm(_PROMPT_B, user, model)


print("Detectors ready.")

Detectors ready.


In [38]:
# ============================================================
# CELL 7 — Detector: System A (no XML) and System B (with XML RAG)
# ============================================================

import json
import time
import re
from openai import OpenAI

try:
    import anthropic
except ImportError:
    !pip install -q anthropic
    import anthropic

CONFIDENCE_THRESHOLD = 0.85

# ── Load credentials from Colab Secrets ──────────────────────
try:
    from google.colab import userdata
    OPENAI_ORG_ID     = userdata.get('OPENAI_ORG_ID')     or ""
    OPENAI_PROJECT_ID = userdata.get('OPENAI_PROJECT_ID') or ""
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY') or ""
except:
    OPENAI_ORG_ID     = ""
    OPENAI_PROJECT_ID = ""
    ANTHROPIC_API_KEY = ""

print(f"OpenAI org ID    : {'set' if OPENAI_ORG_ID     else 'not set'}")
print(f"OpenAI project ID: {'set' if OPENAI_PROJECT_ID else 'not set'}")
print(f"Anthropic key    : {'set' if ANTHROPIC_API_KEY  else 'not set (Claude models will be skipped)'}")

# ── Build opt-out headers for OpenAI ─────────────────────────
_openai_headers = {}
if OPENAI_ORG_ID:
    _openai_headers["OpenAI-Organization"] = OPENAI_ORG_ID
if OPENAI_PROJECT_ID:
    _openai_headers["OpenAI-Project"] = OPENAI_PROJECT_ID

# ── Clients ───────────────────────────────────────────────────
openai_client    = OpenAI(
    api_key         = OPENAI_API_KEY,
    default_headers = _openai_headers,
)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print(f"\nOpenAI client  : set")
print(f"Anthropic client: {'set' if anthropic_client else 'not set (no key)'}")

# ── Models that use Responses API (GPT-5 and newer) ──────────
# GPT-4 and older → Chat Completions API
# GPT-5 and newer → Responses API
_RESPONSES_API_MODELS = {
    "gpt-5", "gpt-5-mini", "gpt-5.1", "gpt-5.4", "gpt-5.4-mini",
    "gpt-5.4-nano", "gpt-5.4-pro", "gpt-5.5", "gpt-5.5-pro",
    "o3", "o3-mini", "o4-mini",
}

_PROMPT_A = """Find duplicate GitHub issues.

Given a QUERY ISSUE and a list of CANDIDATE ISSUES, is the query a duplicate of any candidate?

Two issues are duplicates only if they report the exact same bug or request the exact same change.
Same package or similar topic is not enough.
When in doubt return is_duplicate: false.

Respond with ONLY valid JSON:
{
  "is_duplicate": <bool>,
  "duplicate_of": <issue number as string, or null>,
  "confidence"  : <float 0.0-1.0>
}"""

_PROMPT_B = """Find duplicate GitHub issues.

You will be given DUPLICATE EXAMPLES from this codebase's history, a QUERY ISSUE, and CANDIDATE ISSUES.

The duplicate examples are real pairs of bugs that were marked as the same issue in this project.
Use them to understand what counts as a duplicate in this specific codebase.

Two GitHub issues are duplicates only if they report the exact same bug or request the exact same change.
Same package or similar topic is not enough.
When in doubt return is_duplicate: false.

Respond with ONLY valid JSON:
{
  "is_duplicate": <bool>,
  "duplicate_of": <issue number as string, or null>,
  "confidence"  : <float 0.0-1.0>
}"""


def fmt_github(issue):
    return (f"Issue #{issue['id']}\n"
            f"Title  : {issue['title']}\n"
            f"Package: {issue.get('package', '-')}")


def fmt_xml_pair(bug, dup_bug):
    result  = "--- Example duplicate pair ---\n"
    result += f"Bug A: {bug['title']}\n"
    result += f"  {(bug.get('body') or '')[:200]}\n"
    if dup_bug:
        result += f"Bug B (same issue as A): {dup_bug['title']}\n"
        result += f"  {(dup_bug.get('body') or '')[:200]}\n"
    result += "(These two were marked as duplicates of each other)\n"
    return result


def parse_json(raw):
    """Extract JSON from raw LLM output.
    Handles: markdown fences, preamble text (Claude reasoning), bad escapes.
    Strategy: find ALL {...} blocks and try the last one first (Claude puts
    its JSON at the end after reasoning), then fall back to earlier ones.
    """
    # Strip markdown fences
    raw = re.sub(r"```(?:json)?\s*", "", raw)
    raw = re.sub(r"```", "", raw).strip()

    # Collect all top-level {...} spans
    candidates = []
    depth, start = 0, None
    for idx, ch in enumerate(raw):
        if ch == "{":
            if depth == 0:
                start = idx
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                candidates.append(raw[start:idx+1])
                start = None

    # Try last candidate first (Claude puts JSON after its reasoning),
    # then work backwards
    for blob in reversed(candidates):
        # Fix stray backslashes
        blob = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', blob)
        try:
            return json.loads(blob)
        except json.JSONDecodeError:
            continue

    raise ValueError(f"Could not parse JSON from: {raw[:300]}")


def call_llm(prompt, user, model):
    """Unified LLM call — routes to OpenAI or Anthropic based on model name.
    GPT-4 and older → Chat Completions API
    GPT-5 and newer → Responses API
    Claude           → Anthropic Messages API
    """
    content  = prompt + "\n\n" + user
    provider = "anthropic" if model.startswith("claude") else "openai"
    use_responses_api = any(
        model.startswith(m) for m in _RESPONSES_API_MODELS
    )

    for attempt in range(3):
        try:
            if provider == "anthropic":
                # Anthropic Messages API — NO prefill (not supported on extended
                # thinking models sonnet-4-6 / opus-4-7). parse_json extracts
                # the last JSON block from the response, handling any preamble.
                resp = anthropic_client.messages.create(
                    model      = model,
                    max_tokens = 1024,
                    system     = (
                        "You are a duplicate-detection classifier. "
                        "You MUST end your response with a JSON object on its own line, "
                        "with exactly these keys: is_duplicate (bool), "
                        "duplicate_of (issue number as string, or null), confidence (float 0-1). "
                        "End with exactly this format (values will vary): "
                        "{\"is_duplicate\": false, \"duplicate_of\": null, \"confidence\": 0.1}"
                    ),
                    messages   = [{"role": "user", "content": content}],
                )
                raw = resp.content[0].text

            elif use_responses_api:
                # OpenAI Responses API (GPT-5 and newer)
                resp = openai_client.responses.create(
                    model         = model,
                    input         = content,
                    extra_headers = _openai_headers,
                )
                raw = resp.output_text

            else:
                # OpenAI Chat Completions API (GPT-4 and older)
                resp = openai_client.chat.completions.create(
                    model         = model,
                    messages      = [{"role": "user", "content": content}],
                    temperature   = 0.0,
                    max_tokens    = 300,
                    extra_headers = _openai_headers,
                )
                raw = resp.choices[0].message.content

            # Guard against empty responses
            if not raw or not raw.strip():
                print(f"    Empty response from {model}")
                raise ValueError("Empty response")

            result     = parse_json(raw)
            is_dup     = bool(result.get("is_duplicate", False))
            confidence = float(result.get("confidence", 0.0))
            if is_dup and confidence < CONFIDENCE_THRESHOLD:
                is_dup = False
            result["is_duplicate"] = is_dup
            result["confidence"]   = confidence
            if not is_dup:
                result["duplicate_of"] = None
            return result

        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {type(e).__name__}: {e}")

OpenAI org ID    : set
OpenAI project ID: set
Anthropic key    : set

OpenAI client  : set
Anthropic client: set


In [39]:
from google.colab import userdata

secrets_to_check = [
    'OPENAI_API_KEY',
    'OPENAI_ORG_ID',
    'OPENAI_PROJECT_ID',
    'ANTHROPIC_API_KEY',
]

for name in secrets_to_check:
    try:
        val = userdata.get(name)
        print(f"✅ {name}: {val[:8]}...{val[-4:] if len(val) > 12 else val}")
    except Exception as e:
        print(f"❌ {name}: {e}")

✅ OPENAI_API_KEY: sk-proj-...SLoA
✅ OPENAI_ORG_ID: org-JpCc...hfGN
✅ OPENAI_PROJECT_ID: proj_vKc...GTH1
✅ ANTHROPIC_API_KEY: sk-ant-a...twAA


# System A runs above — results saved per model

In [40]:
# ============================================================
# System A (no RAG) — GPT models only
# ============================================================

import os, json, re, csv
from tqdm.notebook import tqdm

gh_issues_filtered = [
    i for i in gh_issues
    if not re.search(r'bugzilla bug \d+', i['title'], re.IGNORECASE)
]
print(f"Genuine GitHub issues: {len(gh_issues_filtered)}")

summary_a_gpt = []

for model in OPENAI_MODELS:
    model_slug = model.replace("/", "-").replace(".", "_")
    path_json  = f'{RESULTS_DIR}/predictions_A_{model_slug}.json'
    path_csv   = f'{RESULTS_DIR}/predictions_A_{model_slug}.csv'

    print(f"\n{'='*55}")
    print(f"  System A (GPT) — {model}")
    print(f"{'='*55}")

    if os.path.exists(path_json):
        os.remove(path_json)

    preds = []
    for issue in tqdm(gh_issues_filtered, desc=f"A | {model}"):
        try:
            dup_candidates = retriever.get_dup_candidates(issue, k=DUP_CANDIDATE_K)
            pred           = predict_A(issue, dup_candidates, model)
            dup_title      = retriever.gh_by_id.get(
                str(pred.get("duplicate_of") or ""), {}
            ).get("title", "")
            pred.update({
                "model"          : model,
                "issue_id"       : issue["id"],
                "title"          : issue["title"],
                "package"        : issue["package"],
                "duplicate_title": dup_title,
            })
            preds.append(pred)
            with open(path_json, 'w') as f:
                json.dump(preds, f, indent=2)
        except Exception as e:
            print(f"  Skipped #{issue['id']}: {e}")

    if preds:
        keys = ["model","issue_id","title","package","is_duplicate",
                "duplicate_of","duplicate_title","confidence"]
        with open(path_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(preds)

    n_dup = sum(1 for p in preds if p["is_duplicate"])
    print(f"  Processed : {len(preds)} | Duplicates: {n_dup} ({n_dup/max(len(preds),1)*100:.1f}%)")
    print(f"  Saved JSON: {path_json}")
    print(f"  Saved CSV : {path_csv}")
    summary_a_gpt.append({"model": model, "processed": len(preds), "duplicates": n_dup})

print(f"\n{'='*55}")
print(f"  System A GPT Summary")
print(f"{'='*55}")
print(f"  {'Model':35s} {'Processed':10s} {'Duplicates':10s}")
print(f"  {'-'*55}")
for s in summary_a_gpt:
    print(f"  {s['model']:35s} {s['processed']:10d} {s['duplicates']:10d}")


Genuine GitHub issues: 75

  System A (GPT) — gpt-4o-mini


A | gpt-4o-mini:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 2 (2.7%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_A_gpt-4o-mini.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_A_gpt-4o-mini.csv

  System A (GPT) — gpt-4o


A | gpt-4o:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 4 (5.3%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_A_gpt-4o.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_A_gpt-4o.csv

  System A (GPT) — gpt-4.1-mini


...

  Processed : 75 | Duplicates: 11 (14.7%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_A_gpt-5_4-mini.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_A_gpt-5_4-mini.csv

  System A (GPT) — gpt-5.4


A | gpt-5.4:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 6 (8.0%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_A_gpt-5_4.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_A_gpt-5_4.csv

  System A (GPT) — gpt-5.5


A | gpt-5.5:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 4 (5.3%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_A_gpt-5_5.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_A_gpt-5_5.csv

  System A GPT Summary
  Model                               Processed  Duplicates
  -------------------------------------------------------
  gpt-4o-mini                                 75          2
  gpt-4o                                      75          4
  gpt-4.1-mini                                75          4
  gpt-4.1                                     75          3
  gpt-5.4-mini                                75         11
  gpt-5.4                                     75          6
  gpt-5.5                                     75          4


### System A — Claude models

In [41]:
# ============================================================
# System A (no RAG) — Claude models only
# ============================================================

import os, json, re, csv
from tqdm.notebook import tqdm

gh_issues_filtered = [
    i for i in gh_issues
    if not re.search(r'bugzilla bug \d+', i['title'], re.IGNORECASE)
]
print(f"Genuine GitHub issues: {len(gh_issues_filtered)}")

summary_a_claude = []

for model in CLAUDE_MODELS:
    model_slug = model.replace("/", "-").replace(".", "_")
    path_json  = f'{RESULTS_DIR}/predictions_A_{model_slug}.json'
    path_csv   = f'{RESULTS_DIR}/predictions_A_{model_slug}.csv'

    print(f"\n{'='*55}")
    print(f"  System A (Claude) — {model}")
    print(f"{'='*55}")

    if os.path.exists(path_json):
        os.remove(path_json)

    preds = []
    for issue in tqdm(gh_issues_filtered, desc=f"A | {model}"):
        try:
            dup_candidates = retriever.get_dup_candidates(issue, k=DUP_CANDIDATE_K)
            pred           = predict_A(issue, dup_candidates, model)
            dup_title      = retriever.gh_by_id.get(
                str(pred.get("duplicate_of") or ""), {}
            ).get("title", "")
            pred.update({
                "model"          : model,
                "issue_id"       : issue["id"],
                "title"          : issue["title"],
                "package"        : issue["package"],
                "duplicate_title": dup_title,
            })
            preds.append(pred)
            with open(path_json, 'w') as f:
                json.dump(preds, f, indent=2)
        except Exception as e:
            print(f"  Skipped #{issue['id']}: {e}")

    if preds:
        keys = ["model","issue_id","title","package","is_duplicate",
                "duplicate_of","duplicate_title","confidence"]
        with open(path_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(preds)

    n_dup = sum(1 for p in preds if p["is_duplicate"])
    print(f"  Processed : {len(preds)} | Duplicates: {n_dup} ({n_dup/max(len(preds),1)*100:.1f}%)")
    print(f"  Saved JSON: {path_json}")
    print(f"  Saved CSV : {path_csv}")
    summary_a_claude.append({"model": model, "processed": len(preds), "duplicates": n_dup})

print(f"\n{'='*55}")
print(f"  System A Claude Summary")
print(f"{'='*55}")
print(f"  {'Model':35s} {'Processed':10s} {'Duplicates':10s}")
print(f"  {'-'*55}")
for s in summary_a_claude:
    print(f"  {s['model']:35s} {s['processed']:10d} {s['duplicates']:10d}")


Genuine GitHub issues: 75

  System A (Claude) — claude-haiku-4-5


A | claude-haiku-4-5:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 6 (8.0%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_A_claude-haiku-4-5.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_A_claude-haiku-4-5.csv

  System A (Claude) — claude-sonnet-4-6


A | claude-sonnet-4-6:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 2 (2.7%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_A_claude-sonnet-4-6.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_A_claude-sonnet-4-6.csv

  System A (Claude) — claude-opus-4-7


A | claude-opus-4-7:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 2 (2.7%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_A_claude-opus-4-7.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_A_claude-opus-4-7.csv

  System A Claude Summary
  Model                               Processed  Duplicates
  -------------------------------------------------------
  claude-haiku-4-5                            75          6
  claude-sonnet-4-6                           75          2
  claude-opus-4-7                             75          2


# System B runs above — results saved per model

In [42]:
# ============================================================
# System B (with RAG) — GPT models only
# ============================================================

import os, json, re, csv
from tqdm.notebook import tqdm

gh_issues_filtered = [
    i for i in gh_issues
    if not re.search(r'bugzilla bug \d+', i['title'], re.IGNORECASE)
]
print(f"Genuine GitHub issues: {len(gh_issues_filtered)}")

summary_b_gpt = []

for model in OPENAI_MODELS:
    model_slug = model.replace("/", "-").replace(".", "_")
    path_json  = f'{RESULTS_DIR}/predictions_B_{model_slug}.json'
    path_csv   = f'{RESULTS_DIR}/predictions_B_{model_slug}.csv'

    print(f"\n{'='*55}")
    print(f"  System B (GPT) — {model}")
    print(f"{'='*55}")

    if os.path.exists(path_json):
        os.remove(path_json)

    preds = []
    for issue in tqdm(gh_issues_filtered, desc=f"B | {model}"):
        try:
            dup_candidates = retriever.get_dup_candidates(issue, k=DUP_CANDIDATE_K)
            xml_pairs      = retriever.get_xml_dup_pairs(issue,  k=XML_CONTEXT_K)
            pred           = predict_B(issue, dup_candidates, xml_pairs, model)
            dup_title      = retriever.gh_by_id.get(
                str(pred.get("duplicate_of") or ""), {}
            ).get("title", "")
            pred.update({
                "model"          : model,
                "issue_id"       : issue["id"],
                "title"          : issue["title"],
                "package"        : issue["package"],
                "duplicate_title": dup_title,
            })
            preds.append(pred)
            with open(path_json, 'w') as f:
                json.dump(preds, f, indent=2)
        except Exception as e:
            print(f"  Skipped #{issue['id']}: {e}")

    if preds:
        keys = ["model","issue_id","title","package","is_duplicate",
                "duplicate_of","duplicate_title","confidence"]
        with open(path_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(preds)

    n_dup = sum(1 for p in preds if p["is_duplicate"])
    print(f"  Processed : {len(preds)} | Duplicates: {n_dup} ({n_dup/max(len(preds),1)*100:.1f}%)")
    print(f"  Saved JSON: {path_json}")
    print(f"  Saved CSV : {path_csv}")
    summary_b_gpt.append({"model": model, "processed": len(preds), "duplicates": n_dup})

print(f"\n{'='*55}")
print(f"  System B GPT Summary")
print(f"{'='*55}")
print(f"  {'Model':35s} {'Processed':10s} {'Duplicates':10s}")
print(f"  {'-'*55}")
for s in summary_b_gpt:
    print(f"  {s['model']:35s} {s['processed']:10d} {s['duplicates']:10d}")


Genuine GitHub issues: 75

  System B (GPT) — gpt-4o-mini


B | gpt-4o-mini:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 6 (8.0%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_B_gpt-4o-mini.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_B_gpt-4o-mini.csv

  System B (GPT) — gpt-4o


B | gpt-4o:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 19 (25.3%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_B_gpt-4o.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_B_gpt-4o.csv

  System B (GPT) — gpt-4.1-mini


...

  Processed : 75 | Duplicates: 5 (6.7%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_B_gpt-5_4-mini.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_B_gpt-5_4-mini.csv

  System B (GPT) — gpt-5.4


B | gpt-5.4:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 7 (9.3%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_B_gpt-5_4.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_B_gpt-5_4.csv

  System B (GPT) — gpt-5.5


B | gpt-5.5:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 6 (8.0%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_B_gpt-5_5.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_B_gpt-5_5.csv

  System B GPT Summary
  Model                               Processed  Duplicates
  -------------------------------------------------------
  gpt-4o-mini                                 75          6
  gpt-4o                                      75         19
  gpt-4.1-mini                                75          6
  gpt-4.1                                     75          8
  gpt-5.4-mini                                75          5
  gpt-5.4                                     75          7
  gpt-5.5                                     75          6


### System B — Claude models

In [43]:
# ============================================================
# System B (with RAG) — Claude models only
# ============================================================

import os, json, re, csv
from tqdm.notebook import tqdm

gh_issues_filtered = [
    i for i in gh_issues
    if not re.search(r'bugzilla bug \d+', i['title'], re.IGNORECASE)
]
print(f"Genuine GitHub issues: {len(gh_issues_filtered)}")

summary_b_claude = []

for model in CLAUDE_MODELS:
    model_slug = model.replace("/", "-").replace(".", "_")
    path_json  = f'{RESULTS_DIR}/predictions_B_{model_slug}.json'
    path_csv   = f'{RESULTS_DIR}/predictions_B_{model_slug}.csv'

    print(f"\n{'='*55}")
    print(f"  System B (Claude) — {model}")
    print(f"{'='*55}")

    if os.path.exists(path_json):
        os.remove(path_json)

    preds = []
    for issue in tqdm(gh_issues_filtered, desc=f"B | {model}"):
        try:
            dup_candidates = retriever.get_dup_candidates(issue, k=DUP_CANDIDATE_K)
            xml_pairs      = retriever.get_xml_dup_pairs(issue,  k=XML_CONTEXT_K)
            pred           = predict_B(issue, dup_candidates, xml_pairs, model)
            dup_title      = retriever.gh_by_id.get(
                str(pred.get("duplicate_of") or ""), {}
            ).get("title", "")
            pred.update({
                "model"          : model,
                "issue_id"       : issue["id"],
                "title"          : issue["title"],
                "package"        : issue["package"],
                "duplicate_title": dup_title,
            })
            preds.append(pred)
            with open(path_json, 'w') as f:
                json.dump(preds, f, indent=2)
        except Exception as e:
            print(f"  Skipped #{issue['id']}: {e}")

    if preds:
        keys = ["model","issue_id","title","package","is_duplicate",
                "duplicate_of","duplicate_title","confidence"]
        with open(path_csv, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(preds)

    n_dup = sum(1 for p in preds if p["is_duplicate"])
    print(f"  Processed : {len(preds)} | Duplicates: {n_dup} ({n_dup/max(len(preds),1)*100:.1f}%)")
    print(f"  Saved JSON: {path_json}")
    print(f"  Saved CSV : {path_csv}")
    summary_b_claude.append({"model": model, "processed": len(preds), "duplicates": n_dup})

print(f"\n{'='*55}")
print(f"  System B Claude Summary")
print(f"{'='*55}")
print(f"  {'Model':35s} {'Processed':10s} {'Duplicates':10s}")
print(f"  {'-'*55}")
for s in summary_b_claude:
    print(f"  {s['model']:35s} {s['processed']:10d} {s['duplicates']:10d}")


Genuine GitHub issues: 75

  System B (Claude) — claude-haiku-4-5


B | claude-haiku-4-5:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 4 (5.3%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_B_claude-haiku-4-5.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_B_claude-haiku-4-5.csv

  System B (Claude) — claude-sonnet-4-6


B | claude-sonnet-4-6:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 2 (2.7%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_B_claude-sonnet-4-6.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_B_claude-sonnet-4-6.csv

  System B (Claude) — claude-opus-4-7


B | claude-opus-4-7:   0%|          | 0/75 [00:00<?, ?it/s]

  Processed : 75 | Duplicates: 3 (4.0%)
  Saved JSON: /content/drive/MyDrive/dup_detect/results/predictions_B_claude-opus-4-7.json
  Saved CSV : /content/drive/MyDrive/dup_detect/results/predictions_B_claude-opus-4-7.csv

  System B Claude Summary
  Model                               Processed  Duplicates
  -------------------------------------------------------
  claude-haiku-4-5                            75          4
  claude-sonnet-4-6                           75          2
  claude-opus-4-7                             75          3


RAG notes :GitHub issue text
       ↓
   BGE embeds it
       ↓
ChromaDB searches XML files
       ↓
Returns top-3 similar XML bugs
       ↓
Injected into System B prompt as "SIMILAR HISTORICAL BUGS FROM THIS CODEBASE"
       ↓
GPT-4o-mini reads them as context when deciding if it's a duplicate

EMBED_MODEL = "BAAI/bge-base-en-v1.5"

ChromaDB is a vector database that stores embeddings and searches them by similarity.

The few-shot examples are exact labeled pairs — that's the ground truth signal. The RAG context is just topically similar bugs — that's the domain knowledge signal. Both go into the prompt but serve different purposes.

# Cell 9 — Evaluation Block
Evaluates all model predictions against a **zero-duplicates ground truth** (conservative baseline). Computes Precision, Recall, F1 Score, and Accuracy per model/system and saves results to CSV.

In [44]:
# ============================================================
# CELL 9 — Evaluation (Ground Truth: No Duplicates)
# ============================================================
# Since we have no labeled ground truth for this dataset,
# we use the conservative assumption: ALL issues are non-duplicates.
# Under this assumption:
#   - True Positives  (TP) = 0  (no real duplicates to find)
#   - False Positives (FP) = predictions flagged as duplicate
#   - True Negatives  (TN) = issues correctly NOT flagged
#   - False Negatives (FN) = 0  (nothing was actually duplicate)
#
# Metrics derived:
#   Precision = TP / (TP + FP)  → 0 if any FP, NaN if no predictions (set to 1.0)
#   Recall    = TP / (TP + FN)  → always 1.0 (nothing to miss)
#   F1        = 2 * P * R / (P + R)
#   Accuracy  = (TP + TN) / Total
# ============================================================

import os
import json
import csv
import glob
import pandas as pd
from pathlib import Path

# ── Settings ────────────────────────────────────────────────
EVAL_OUTPUT_CSV = f'{RESULTS_DIR}/evaluation_no_duplicates_gt.csv'

# ── Gather all prediction CSVs ──────────────────────────────
pred_files = sorted(glob.glob(f'{RESULTS_DIR}/predictions_*.csv'))
print(f"Found {len(pred_files)} prediction files:")
for f in pred_files:
    print(f"  {os.path.basename(f)}")

if not pred_files:
    print("\n⚠ No prediction CSVs found in RESULTS_DIR. Run Cells 8A and 8B first.")
else:
    rows = []

    for fpath in pred_files:
        fname = os.path.basename(fpath)          # e.g. predictions_A_gpt-4o-mini.csv
        # Parse system (A or B) and model slug from filename
        parts = fname.replace("predictions_", "").replace(".csv", "").split("_", 1)
        system    = parts[0]                     # "A" or "B"
        model_slug = parts[1] if len(parts) > 1 else "unknown"

        df = pd.read_csv(fpath)
        total = len(df)

        if total == 0:
            print(f"  Skipping {fname} — empty file")
            continue

        # Under "no duplicates" ground truth:
        #   Ground truth label for every issue = False (not a duplicate)
        n_flagged  = int(df["is_duplicate"].sum())   # FP count
        n_negative = total - n_flagged               # TN count

        TP = 0
        FP = n_flagged
        TN = n_negative
        FN = 0

        # Precision: if no predictions → 1.0 (no false alarms = perfect precision)
        precision = 1.0 if (TP + FP) == 0 else TP / (TP + FP)
        recall    = 1.0  # TP / (TP + FN) = 0 / 0 → defined as 1.0 (nothing to recall)
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)
        accuracy  = (TP + TN) / total

        # Average confidence of flagged predictions
        if n_flagged > 0:
            avg_conf_flagged = float(df.loc[df["is_duplicate"] == True, "confidence"].mean())
        else:
            avg_conf_flagged = float("nan")

        rows.append({
            "system"               : system,
            "model"                : model_slug.replace("_", "."),
            "total_issues"         : total,
            "flagged_as_duplicate" : n_flagged,
            "true_positives"       : TP,
            "false_positives"      : FP,
            "true_negatives"       : TN,
            "false_negatives"      : FN,
            "precision"            : round(precision, 4),
            "recall"               : round(recall,    4),
            "f1_score"             : round(f1,        4),
            "accuracy"             : round(accuracy,  4),
            "avg_confidence_flagged": round(avg_conf_flagged, 4) if not pd.isna(avg_conf_flagged) else None,
        })

    # ── Save to CSV ─────────────────────────────────────────
    eval_df = pd.DataFrame(rows).sort_values(["system", "model"])
    eval_df.to_csv(EVAL_OUTPUT_CSV, index=False)
    print(f"\n Evaluation saved to: {EVAL_OUTPUT_CSV}")

    # ── Pretty-print results ─────────────────────────────────
    print("\n" + "="*75)
    print("  EVALUATION RESULTS  (Ground Truth: 0 duplicates in dataset)")
    print("="*75)
    print(f"  {'System':<8} {'Model':<30} {'Flagged':>8} {'Precision':>10} {'Recall':>8} {'F1':>8} {'Accuracy':>10}")
    print(f"  {'-'*75}")
    for _, r in eval_df.iterrows():
        print(f"  {r['system']:<8} {r['model']:<30} {r['flagged_as_duplicate']:>8} "
              f"{r['precision']:>10.4f} {r['recall']:>8.4f} {r['f1_score']:>8.4f} {r['accuracy']:>10.4f}")
    print("="*75)

    print("\nInterpretation:")
    print("  Precision = 0.0 means all flagged predictions are false positives (model is over-predicting).")
    print("  Precision = 1.0 means the model flagged nothing (conservative — no false alarms).")
    print("  Accuracy reflects the fraction of issues correctly classified as non-duplicate.")
    print("  F1 balances precision and recall; under this GT, a model that flags nothing scores 1.0.")
    print("  Avg Confidence shows how sure the model was when it did flag something.")


Found 20 prediction files:
  predictions_A_claude-haiku-4-5.csv
  predictions_A_claude-opus-4-7.csv
  predictions_A_claude-sonnet-4-6.csv
  predictions_A_gpt-4_1-mini.csv
  predictions_A_gpt-4_1.csv
  predictions_A_gpt-4o-mini.csv
  predictions_A_gpt-4o.csv
  predictions_A_gpt-5_4-mini.csv
  predictions_A_gpt-5_4.csv
  predictions_A_gpt-5_5.csv
  predictions_B_claude-haiku-4-5.csv
  predictions_B_claude-opus-4-7.csv
  predictions_B_claude-sonnet-4-6.csv
  predictions_B_gpt-4_1-mini.csv
  predictions_B_gpt-4_1.csv
  predictions_B_gpt-4o-mini.csv
  predictions_B_gpt-4o.csv
  predictions_B_gpt-5_4-mini.csv
  predictions_B_gpt-5_4.csv
  predictions_B_gpt-5_5.csv

 Evaluation saved to: /content/drive/MyDrive/dup_detect/results/evaluation_no_duplicates_gt.csv

  EVALUATION RESULTS  (Ground Truth: 0 duplicates in dataset)
  System   Model                           Flagged  Precision   Recall       F1   Accuracy
  ---------------------------------------------------------------------------
  A 

In [45]:
# ============================================================
# CELL 9 — Evaluation (Ground Truth: Zero Duplicates)
# ============================================================
# Assumption: ALL 75 issues are genuine — zero duplicates exist.
# Every flag the model makes is therefore a False Positive.
#
# CLASS DEFINITIONS:
#   Positive class (1) = duplicate      → ground truth: NONE exist
#   Negative class (0) = not duplicate  → ground truth: ALL are here
#
#   TP = 0          (no real duplicates to correctly find)
#   FP = #flagged   (all flags are wrong under this GT)
#   TN = total - #flagged
#   FN = 0          (nothing real to miss)
#
# PER-CLASS METRICS:
#   Precision (positive) = TP/(TP+FP) = 0  if flagged>0, else 1.0
#   Recall    (positive) = TP/(TP+FN) = 1.0 (defined; nothing real to miss)
#   Precision (negative) = TN/(TN+FN) = 1.0 always (FN=0)
#   Recall    (negative) = TN/(TN+FP) = TN/total
#
# MACRO AVERAGES:
#   Macro Precision = (Prec_pos + Prec_neg) / 2
#   Macro Recall    = (Rec_pos  + Rec_neg)  / 2
#   Macro F1        = (F1_pos   + F1_neg)   / 2
#   Accuracy        = (TP + TN) / total = TN / total
# ============================================================

import os, glob
import pandas as pd

EVAL_OUTPUT_CSV = f'{RESULTS_DIR}/evaluation_no_duplicates_gt.csv'

pred_files = sorted(glob.glob(f'{RESULTS_DIR}/predictions_*.csv'))
print(f"Found {len(pred_files)} prediction files:")
for f in pred_files:
    print(f"  {os.path.basename(f)}")

if not pred_files:
    print("\n  No prediction CSVs found. Run Cells 8A and 8B first.")
else:
    rows = []

    for fpath in pred_files:
        fname      = os.path.basename(fpath)
        parts      = fname.replace("predictions_", "").replace(".csv", "").split("_", 1)
        system     = parts[0]
        model_slug = parts[1] if len(parts) > 1 else "unknown"

        df    = pd.read_csv(fpath)
        total = len(df)
        if total == 0:
            print(f"  Skipping {fname} — empty")
            continue

        # Cast is_duplicate safely
        df["is_duplicate"] = df["is_duplicate"].apply(
            lambda x: str(x).strip().lower() in ("true", "1", "yes")
        )

        n_flagged = int(df["is_duplicate"].sum())
        TP = 0
        FP = n_flagged
        TN = total - n_flagged
        FN = 0

        # ── Positive class (duplicate) ──────────────────────
        prec_pos = 1.0 if (TP + FP) == 0 else TP / (TP + FP)   # 1.0 if nothing flagged, else 0.0
        rec_pos  = 1.0                                            # TP/(TP+FN): defined as 1.0
        f1_pos   = (2 * prec_pos * rec_pos / (prec_pos + rec_pos)
                    if (prec_pos + rec_pos) > 0 else 0.0)

        # ── Negative class (not duplicate) ──────────────────
        prec_neg = 1.0                                            # TN/(TN+FN): FN=0 always
        rec_neg  = TN / total if total > 0 else 0.0              # TN/(TN+FP)
        f1_neg   = (2 * prec_neg * rec_neg / (prec_neg + rec_neg)
                    if (prec_neg + rec_neg) > 0 else 0.0)

        # ── Macro averages ───────────────────────────────────
        macro_precision = (prec_pos + prec_neg) / 2
        macro_recall    = (rec_pos  + rec_neg)  / 2
        macro_f1        = (f1_pos   + f1_neg)   / 2
        accuracy        = TN / total

        avg_conf = (float(df.loc[df["is_duplicate"], "confidence"].mean())
                    if n_flagged > 0 else None)

        rows.append({
            "system"                 : system,
            "model"                  : model_slug.replace("_", "."),
            "total_issues"           : total,
            "flagged_as_duplicate"   : n_flagged,
            "TP"                     : TP,
            "FP"                     : FP,
            "TN"                     : TN,
            "FN"                     : FN,
            "precision_positive"     : round(prec_pos,          4),
            "recall_positive"        : round(rec_pos,           4),
            "f1_positive"            : round(f1_pos,            4),
            "precision_negative"     : round(prec_neg,          4),
            "recall_negative"        : round(rec_neg,           4),
            "f1_negative"            : round(f1_neg,            4),
            "macro_precision"        : round(macro_precision,   4),
            "macro_recall"           : round(macro_recall,      4),
            "macro_f1"               : round(macro_f1,          4),
            "accuracy"               : round(accuracy,          4),
            "avg_confidence_flagged" : round(avg_conf, 4) if avg_conf is not None else "",
        })

    eval_df = pd.DataFrame(rows).sort_values(["system", "model"])
    eval_df.to_csv(EVAL_OUTPUT_CSV, index=False)
    print(f"\n Evaluation saved → {EVAL_OUTPUT_CSV}")

    # ── Pretty print ─────────────────────────────────────────
    sep = "=" * 105
    print("\n" + sep)
    print("  EVALUATION RESULTS  (Ground Truth: 0 duplicates in dataset)")
    print(sep)
    print(f"  {'Sys':<4} {'Model':<26} {'Flag':>5} "
          f"{'P+':>7} {'R+':>7} {'F1+':>7} "
          f"{'P-':>7} {'R-':>7} {'F1-':>7} "
          f"{'Mac-P':>7} {'Mac-R':>7} {'Mac-F1':>7} {'Acc':>7}")
    print("  " + "-" * 103)
    for _, r in eval_df.iterrows():
        print(f"  {r['system']:<4} {r['model']:<26} {r['flagged_as_duplicate']:>5} "
              f"{r['precision_positive']:>7.4f} {r['recall_positive']:>7.4f} {r['f1_positive']:>7.4f} "
              f"{r['precision_negative']:>7.4f} {r['recall_negative']:>7.4f} {r['f1_negative']:>7.4f} "
              f"{r['macro_precision']:>7.4f} {r['macro_recall']:>7.4f} {r['macro_f1']:>7.4f} "
              f"{r['accuracy']:>7.4f}")
    print(sep)
    print("\nColumn guide:")
    print("  P+/R+/F1+ = Precision / Recall / F1 for the DUPLICATE (positive) class")
    print("  P-/R-/F1- = Precision / Recall / F1 for the NON-DUPLICATE (negative) class")
    print("  Mac-P/R/F1 = Macro-averaged across both classes")
    print("  Acc        = Fraction of issues correctly classified as non-duplicate")
    print("  Flag       = Number of issues flagged as duplicate by the model")


Found 20 prediction files:
  predictions_A_claude-haiku-4-5.csv
  predictions_A_claude-opus-4-7.csv
  predictions_A_claude-sonnet-4-6.csv
  predictions_A_gpt-4_1-mini.csv
  predictions_A_gpt-4_1.csv
  predictions_A_gpt-4o-mini.csv
  predictions_A_gpt-4o.csv
  predictions_A_gpt-5_4-mini.csv
  predictions_A_gpt-5_4.csv
  predictions_A_gpt-5_5.csv
  predictions_B_claude-haiku-4-5.csv
  predictions_B_claude-opus-4-7.csv
  predictions_B_claude-sonnet-4-6.csv
  predictions_B_gpt-4_1-mini.csv
  predictions_B_gpt-4_1.csv
  predictions_B_gpt-4o-mini.csv
  predictions_B_gpt-4o.csv
  predictions_B_gpt-5_4-mini.csv
  predictions_B_gpt-5_4.csv
  predictions_B_gpt-5_5.csv

 Evaluation saved → /content/drive/MyDrive/dup_detect/results/evaluation_no_duplicates_gt.csv

  EVALUATION RESULTS  (Ground Truth: 0 duplicates in dataset)
  Sys  Model                       Flag      P+      R+     F1+      P-      R-     F1-   Mac-P   Mac-R  Mac-F1     Acc
  -----------------------------------------------------